# Flickr30k final snapshot v2

Cập nhật snapshot cuối mà không thay model triển khai đã khóa.

## Kaggle input bắt buộc

1. `F30K_Final_Audit`
2. `F30K_T1S1_Hybrid_Closeout`
3. `F30K_Hybrid_Audit`
4. `F30K_Hybrid_Paper_Tables`

Runtime: **CPU / Accelerator None / Internet Off**. Notebook không đọc ảnh,
không đọc Held-out Test, không huấn luyện và không xuất ONNX/INT8.

## Kiểm toán và xuất artifact

In [1]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_Final_Snapshot_V2"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def unique_named(name):
    matches = sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(p) for p in matches])
    return matches[0]

def write_json(path, payload):
    temporary = Path(path).with_suffix(Path(path).suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

paths = {
    "final_audit_v1": unique_named("F30K_Final_Audit_RESULT.json"),
    "hybrid_closeout": unique_named("F30K_T1S1_Hybrid_Closeout_RESULT.json"),
    "hybrid_audit": unique_named("F30K_Hybrid_Audit_RESULT.json"),
    "hybrid_paper": unique_named("F30K_Hybrid_Paper_Tables_RESULT.json"),
}
objects = {name: read_json(path) for name, path in paths.items()}
final_v1 = objects["final_audit_v1"]
closeout = objects["hybrid_closeout"]
hybrid_audit = objects["hybrid_audit"]
hybrid_paper = objects["hybrid_paper"]

# The locked historical Final Audit uses the terminal status "passed" rather
# than "complete". Accept only these two successful terminal spellings while
# still requiring its scientific-completion gate below.
assert final_v1["status"] in {"passed", "complete"}
assert final_v1["scientific_evaluation_complete"] is True
assert hybrid_audit["status"] == "passed_closed_negative_result"
assert hybrid_audit["decision"] == "STOP_VALIDATION_NO_GAIN"
assert hybrid_paper["status"] == "complete_negative_result_documented"
assert hybrid_paper["model_registry_change_authorized"] is False
assert closeout["existing_locked_deployment_model_unchanged"] is True
assert closeout["onnx_or_int8_authorized_for_hybrid"] is False
assert closeout["test_set_used_by_followup"] is False

remaining = list(final_v1.get("remaining_external_work", []))
hybrid_note = "Document Hybrid T1S1 negative Validation result in paper/report"
if hybrid_note not in remaining:
    remaining.append(hybrid_note)

result_payload = {
    "schema_version": "f30k_final_snapshot_v2",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "selected_pair": final_v1["selected_pair"],
    "selected_method": final_v1["selected_method"],
    "checkpoint_sha256": final_v1["checkpoint_sha256"],
    "held_out_metrics_x100": final_v1["held_out_metrics_x100"],
    "held_out_test_reopened": False,
    "hybrid_followup": {
        "pair": "T1-S1",
        "method": "Hybrid Sequence KD + Dense Mapped Logit-KD",
        "study_status": "post_hoc_validation_only",
        "mean_CIDEr_D_x100": hybrid_audit["mean_CIDEr_D_x100"],
        "mean_delta_CIDEr_D_x100": hybrid_audit["mean_delta_CIDEr_D_x100"],
        "ci95_delta_CIDEr_D_x100": hybrid_audit["combined_ci95_delta_CIDEr_D_x100"],
        "decision": "STOP_VALIDATION_NO_GAIN",
        "test_set_used": False,
        "onnx_or_int8_authorized": False,
        "model_registry_change_authorized": False,
    },
    "deployment_model_unchanged": closeout["existing_locked_model"],
    "scientific_evaluation_complete": True,
    "kaggle_compute_complete": True,
    "remaining_external_work": remaining,
    "source_sha256": {name: sha256(path) for name, path in paths.items()},
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, result_payload)

report_path = WORK_ROOT / "F30K_FINAL_SNAPSHOT_V2.md"
report_path.write_text(
    "# Flickr30k final snapshot v2\n\n"
    f"- Selected deployment model remains: **{result_payload['deployment_model_unchanged']}**.\n"
    "- Hybrid T1→S1: **STOP_VALIDATION_NO_GAIN**.\n"
    f"- Hybrid mean CIDEr-D: **{hybrid_audit['mean_CIDEr_D_x100']:.2f}**.\n"
    f"- Hybrid mean Δ CIDEr-D: **{hybrid_audit['mean_delta_CIDEr_D_x100']:+.2f}**.\n"
    "- Held-out Test was not reopened.\n"
    "- Hybrid ONNX/INT8 and registry replacement are not authorized.\n",
    encoding="utf-8",
)
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_final_snapshot_v2_manifest",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": [
        {
            "relative_path": result_path.name,
            "size_bytes": result_path.stat().st_size,
            "sha256": sha256(result_path),
        },
        {
            "relative_path": report_path.name,
            "size_bytes": report_path.stat().st_size,
            "sha256": sha256(report_path),
        },
    ],
    "held_out_test_reopened": False,
})

print("Snapshot status         : COMPLETE")
print("Selected model          :", result_payload["deployment_model_unchanged"])
print("Hybrid decision         : STOP_VALIDATION_NO_GAIN")
print("Held-out Test reopened  : False")
print("Registry change         : NOT AUTHORIZED")

Snapshot status         : COMPLETE
Selected model          : T3S3 Direct Logit-KD INT8
Hybrid decision         : STOP_VALIDATION_NO_GAIN
Held-out Test reopened  : False
Registry change         : NOT AUTHORIZED
